## Install necessary libraries

In [2]:
!pip install sentence_transformers -q
!pip install pinecone -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 25.0 MB/s eta 0:00:00


## Load data

In [3]:
cricket_news = """
The T20 World Cup 2024 is in full swing, bringing excitement and drama to cricket fans worldwide.
India's team, captained by Rohit Sharma, is preparing for a crucial match against Ireland, with standout player Jasprit Bumrah expected to play a pivotal role in their campaign.
The tournament has already seen controversy, particularly concerning the pitch conditions at Nassau County International Cricket Stadium in New York, which came under fire after a low-scoring game between Sri Lanka and South Africa.
"""

football_news = """
The world of football is buzzing with excitement as major tournaments and league matches continue to captivate fans globally.
In the UEFA Champions League, the semi-final matchups have been set, with defending champions Real Madrid set to face Manchester City, while Bayern Munich will take on Paris Saint-Germain.
Both ties promise thrilling encounters, featuring some of the best talents in world football.
"""

election_news = """
As election season heats up, the latest developments reveal a highly competitive atmosphere across several key races.
The presidential election has seen intense campaigning from all major candidates, with recent polls indicating a tight race.
Incumbent President Jane Doe is seeking re-election on a platform of economic stability and healthcare reform, while her main rival, Senator John Smith, focuses on education and climate change initiatives."""


ai_revolution_news = """
The AI revolution continues to transform industries and reshape the global economy.
Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.
Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety."""

## Perform embeddings on data

In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
embeddings = embedding_model.encode([cricket_news, football_news, election_news, ai_revolution_news])

In [6]:
embeddings

array([[-0.02901845,  0.01924439, -0.01814239, ...,  0.00644325,
        -0.01740814, -0.01381658],
       [-0.00384664, -0.07271518, -0.00284145, ..., -0.02027755,
         0.0212385 , -0.03015987],
       [-0.02962372,  0.05711373,  0.01119959, ...,  0.0131924 ,
         0.02634867,  0.01807422],
       [-0.01667612,  0.05068189, -0.05662728, ..., -0.00878626,
        -0.02318501, -0.04949613]], dtype=float32)

In [7]:
len(embeddings[0])

768

## Initiate Pinecone

In [8]:
from pinecone import Pinecone
from pinecone import ServerlessSpec

pc = Pinecone(api_key="pcsk_4r2yr1_CZaV2DLqs1jrWc161LibuFSC6L8XUz8FrYeYYiRKQCFisq7pTsWAghV11FGyqxm")
spec = ServerlessSpec(cloud='aws', region='us-east-1')

## Create Index

In [9]:
pc.create_index("example-index", dimension=768, metric="cosine", spec=spec)

{
    "name": "example-index",
    "metric": "cosine",
    "host": "example-index-6rqbdt6.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "

In [10]:
pc.list_indexes()

[
    {
        "name": "example-index",
        "metric": "cosine",
        "host": "example-index-6rqbdt6.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 768,
        "deletion_protection": "disabled",
        "tags": null
    }
]

## Use Index

In [11]:
index = pc.Index("example-index")

In [12]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '150',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 28 Mar 2026 15:14:28 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '37',
                                    'x-pinecone-request-latency-ms': '36',
                                    'x-pinecone-response-duration-ms': '38'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

## Add data to Pinecone Index

In [13]:
index.upsert([
    {"id":"id1", "values":embeddings[0], "metadata":{'source': 'cricket'}},
    {"id":"id2", "values":embeddings[1], "metadata":{'source': 'football'}},
    {"id":"id3", "values":embeddings[2], "metadata":{'source': 'election'}},
    {"id":"id4", "values":embeddings[3], "metadata":{'source': 'ai_revolution'}}
])

UpsertResponse(upserted_count=4, _response_info={'raw_headers': {'date': 'Sat, 28 Mar 2026 15:14:29 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '12392', 'x-pinecone-request-latency-ms': '418', 'x-envoy-upstream-service-time': '322', 'x-pinecone-response-duration-ms': '420', 'grpc-status': '0', 'server': 'envoy'}})

In [14]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '181',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 28 Mar 2026 15:14:29 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '38',
                                    'x-pinecone-request-latency-ms': '37',
                                    'x-pinecone-response-duration-ms': '39'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 4}},
 'storageFullness': 0.0,
 'total_vector_count': 4,
 'vector_type': 'dense'}

## Similarity Search

In [15]:
query = "technology"
query_embedding = embedding_model.encode(query).tolist()

In [16]:
len(query_embedding)

768

In [17]:
similar_docs = index.query(vector=query_embedding, top_k=2, include_metadata=True)
similar_docs

QueryResponse(matches=[{'id': 'id4',
 'metadata': {'source': 'ai_revolution'},
 'score': 0.218233109,
 'values': []}, {'id': 'id1',
 'metadata': {'source': 'cricket'},
 'score': 0.099719055,
 'values': []}], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Sat, 28 Mar 2026 15:14:29 GMT', 'content-type': 'application/json', 'content-length': '225', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '38', 'x-envoy-upstream-service-time': '41', 'x-pinecone-response-duration-ms': '44', 'grpc-status': '0', 'server': 'envoy'}})

## CRUD operations on Vector Database

#### Add data

In [18]:
blockchain_news = """
The blockchain industry continues to evolve rapidly, marked by significant technological advancements and regulatory developments.
This month, the spotlight is on the launch of Ethereum 3.0, which promises enhanced scalability and security features.
This upgrade is expected to drastically reduce transaction fees and increase processing speeds, making decentralized applications (dApps) more efficient and user-friendly.
"""

In [19]:
embedding_query = embedding_model.encode(blockchain_news).tolist()

In [20]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '181',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 28 Mar 2026 15:14:30 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-latency-ms': '2',
                                    'x-pinecone-response-duration-ms': '4'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 4}},
 'storageFullness': 0.0,
 'total_vector_count': 4,
 'vector_type': 'dense'}

In [21]:
index.upsert([{"id":"id5", "values":embedding_query, "metadata":{"source":"blockchain"}}])

UpsertResponse(upserted_count=1, _response_info={'raw_headers': {'date': 'Sat, 28 Mar 2026 15:14:30 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '2', 'x-pinecone-request-logical-size': '3099', 'x-pinecone-request-latency-ms': '160', 'x-envoy-upstream-service-time': '164', 'x-pinecone-response-duration-ms': '165', 'grpc-status': '0', 'server': 'envoy'}})

In [22]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '181',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 28 Mar 2026 15:14:30 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-latency-ms': '2',
                                    'x-pinecone-response-duration-ms': '4'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 5}},
 'storageFullness': 0.0,
 'total_vector_count': 5,
 'vector_type': 'dense'}

In [23]:
query_embedding = embedding_model.encode("technology").tolist()
similar_docs = index.query(vector=query_embedding, top_k=2, include_metadata=True)
similar_docs

QueryResponse(matches=[{'id': 'id4',
 'metadata': {'source': 'ai_revolution'},
 'score': 0.218233109,
 'values': []}, {'id': 'id5',
 'metadata': {'source': 'blockchain'},
 'score': 0.133591652,
 'values': []}], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Sat, 28 Mar 2026 15:14:30 GMT', 'content-type': 'application/json', 'content-length': '228', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '2', 'x-pinecone-request-latency-ms': '36', 'x-envoy-upstream-service-time': '37', 'x-pinecone-response-duration-ms': '43', 'grpc-status': '0', 'server': 'envoy'}})

#### Read data

In [24]:
results = index.fetch(ids=['id1', 'id3'])

In [25]:
for k in results["vectors"]:
  print(results["vectors"][k]['metadata'])

{'source': 'election'}
{'source': 'cricket'}


#### Update data

In [26]:
embedding_query = embedding_model.encode("This is sample document about generative AI").tolist()
index.upsert([("id3", embedding_query, {"source":"gen ai"})])

UpsertResponse(upserted_count=1, _response_info={'raw_headers': {'date': 'Sat, 28 Mar 2026 15:14:30 GMT', 'content-type': 'application/json', 'content-length': '19', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '3', 'x-pinecone-request-logical-size': '3095', 'x-pinecone-request-latency-ms': '136', 'x-envoy-upstream-service-time': '137', 'x-pinecone-response-duration-ms': '138', 'grpc-status': '0', 'server': 'envoy'}})

#### Delete data

In [27]:
index.fetch(ids=['id2'])

FetchResponse(namespace='', vectors={'id2': Vector(id='id2', values=[-0.0038466386, -0.0727151781, -0.00284144678, 0.0574586317, -0.00515250349, -0.0170758758, -0.106182262, -0.0312844031, -0.0369069688, -0.0311232638, -0.020336451, 0.00393040339, 0.0028216159, -0.032265, 0.0612160414, -0.0162303336, -0.00738589885, -0.025414085, -0.0270358194, -0.0188592877, 0.0269980654, 0.0239736009, 0.0206892528, -0.00475601666, 0.0217966773, -0.0107850833, -0.00940124132, 0.0341491215, 0.0194722712, -0.0329037756, -0.00297042588, 0.00035782237, 0.0314517505, 4.502507e-05, 2.13730755e-06, -0.060292054, -0.0257892758, -0.0438521281, 0.0148153612, -0.0247503091, 0.0146588925, -0.0180866215, -0.013675875, -0.0262692682, 0.0120234741, 0.0341221094, -0.0325539224, 0.00870313682, -0.005715237, 0.0301483683, 0.011243714, -0.0319951177, -0.0130925681, -0.00541229686, -0.0530395173, -0.051049713, 0.0548728853, -0.0158240683, -0.0467207693, 0.0122886356, 0.0113953119, -0.0501406491, -0.0139560224, -0.0027249

In [28]:
index.delete(ids=['id2'])

{'_response_info': {'raw_headers': {'date': 'Sat, 28 Mar 2026 15:14:31 GMT',
   'content-type': 'application/json',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-pinecone-request-lsn': '4',
   'x-pinecone-request-logical-size': '0',
   'x-pinecone-request-latency-ms': '134',
   'x-envoy-upstream-service-time': '134',
   'x-pinecone-response-duration-ms': '135',
   'grpc-status': '0',
   'server': 'envoy'}}}

In [29]:
index.fetch(ids=['id2'])

FetchResponse(namespace='', vectors={}, usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Sat, 28 Mar 2026 15:14:31 GMT', 'content-type': 'application/json', 'content-length': '53', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '0', 'x-envoy-upstream-service-time': '1', 'x-pinecone-response-duration-ms': '2', 'grpc-status': '0', 'server': 'envoy'}})

## Delete Pinecone Index

In [30]:
pc.delete_index('example-index')